# 01 · Ingesta y Exploración
## Segmentación Inteligente de Pacientes — Programa de Medicina Funcional · Comfama

| | |
|---|---|
| **Propósito** | Cargar el formulario histórico, aislar el bloque de variables de estilo de vida (**columnas L → AB**) y evaluar su calidad antes de modelar. |
| **Entrada** | Excel `Preguntas y respuestas medico.xlsx`, hoja `DATOS ` (Volume de Unity Catalog). |
| **Salida** | Tabla Delta `<catalog>.<schema>.fenotipo_cardio_ingreso_raw`. |
| **Siguiente paso** | `02_preprocesamiento`. |

---
### 🔎 Contexto y trazabilidad del alcance
El bloque **L:AB** corresponde a las preguntas de **hábitos de alimentación, actividad física, gestión del estrés y medicamentos** del formato de ingreso.

> **Hallazgo validado sobre los datos:** este bloque está diligenciado **casi exclusivamente** para el grupo **Cardiometabólico** — de **4.170** registros con datos, **~94% son Cardiometabólico** (3.928), 206 Digestivo y 36 Mixto (datos parciales).
>
> **Implicación:** el modelo de *clustering* descubrirá **subgrupos / fenotipos emergentes dentro de una cohorte predominantemente cardiometabólica**; **no** reproduce la separación Cardiometabólico / Digestivo / Mixto (cada grupo tiene su propio bloque de columnas en el Excel).

### Paso 0 · Dependencias del entorno
`openpyxl` es necesario para que *pandas* lea archivos `.xlsx`. Reiniciamos el intérprete de Python para que la librería quede disponible en la sesión.

In [ ]:
%pip install openpyxl==3.1.5

In [ ]:
%restart_python

### Paso 1 · Parámetros del notebook (widgets)
Parametrizamos rutas y destino para **reproducibilidad** y para reutilizar el notebook en distintos entornos (dev / prod) sin editar código.

In [ ]:
dbutils.widgets.text("input_path", "/Volumes/main/fenotipo/raw/Preguntas y respuestas medico.xlsx", "Ruta del Excel (Volume)")
dbutils.widgets.text("sheet_name", "DATOS ", "Nombre de la hoja")
dbutils.widgets.text("catalog", "main", "Catálogo")
dbutils.widgets.text("schema", "fenotipo", "Esquema")

INPUT_PATH = dbutils.widgets.get("input_path")
SHEET_NAME = dbutils.widgets.get("sheet_name")   # ⚠️ incluye el espacio final: "DATOS "
CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
RAW_TABLE = f"{CATALOG}.{SCHEMA}.fenotipo_cardio_ingreso_raw"
print("Tabla destino:", RAW_TABLE)

### Paso 2 · Mapa de columnas (trazabilidad Excel → modelo)
Seleccionamos por **posición** (más robusto que por nombre, dado el *encoding* del Excel) y renombramos a `snake_case`. Esta tabla es la **fuente de trazabilidad** entre cada columna original y su variable en el modelo.

| Excel | Variable | Rol |
|---|---|---|
| L | `habitos_alim_texto` | texto crudo (se descarta, redundante) |
| M | `habitos_alim_cantidad` | numérica |
| N–S | `hab_*` (6) | binarias 0/1 |
| T | `actividad_tipo` | categórica |
| U | `actividad_duracion` | categórica |
| V | `actividad_frecuencia` | categórica |
| W | `estres_altas_cargas` | categórica Sí/No |
| X | `estres_tecnica_manejo` | categórica Sí/No |
| Y | `medicamentos_texto` | texto crudo (se descarta, redundante) |
| Z, AA, AB | `med_hipoglicemiantes`, `med_hipolipemiantes`, `med_ninguno` | binarias 0/1 |

Además conservamos `id` (col A), `fecha_atencion` (col D) y `grupos` (col E) como **referencia y control**.

In [ ]:
# Posición 0-based de cada columna del Excel
COL_ID, COL_FECHA, COL_GRUPOS = 0, 3, 4        # A, D, E
FEATURE_POS = list(range(11, 28))              # L(11) .. AB(27)

RENAME_BY_POS = {
    11: "habitos_alim_texto",        # L  (texto crudo -> se descarta luego)
    12: "habitos_alim_cantidad",     # M  numérico 0-5
    13: "hab_dejar_endulzar",        # N  binaria
    14: "hab_dejar_reposteria",      # O  binaria
    15: "hab_incorporar_vegetales",  # P  binaria
    16: "hab_ajustar_carbohidratos", # Q  binaria
    17: "hab_aumentar_proteina",     # R  binaria
    18: "hab_otros",                 # S  binaria
    19: "actividad_tipo",            # T  categórica
    20: "actividad_duracion",        # U  categórica
    21: "actividad_frecuencia",      # V  categórica
    22: "estres_altas_cargas",       # W  Sí/No
    23: "estres_tecnica_manejo",     # X  Sí/No
    24: "medicamentos_texto",        # Y  (texto crudo -> se descarta luego)
    25: "med_hipoglicemiantes",      # Z  binaria
    26: "med_hipolipemiantes",       # AA binaria
    27: "med_ninguno",               # AB binaria
}

### Paso 3 · Lectura del Excel
Cargamos la hoja completa con *pandas* + *openpyxl* y seleccionamos únicamente las columnas de interés (identificadores + bloque L:AB).

In [ ]:
import pandas as pd

pdf_full = pd.read_excel(INPUT_PATH, sheet_name=SHEET_NAME, engine="openpyxl")
print(f"Filas totales en la hoja: {len(pdf_full):,} | Columnas: {pdf_full.shape[1]}")

keep_pos = [COL_ID, COL_FECHA, COL_GRUPOS] + FEATURE_POS
pdf = pdf_full.iloc[:, keep_pos].copy()

rename_map = {pdf_full.columns[p]: name for p, name in RENAME_BY_POS.items()}
rename_map[pdf_full.columns[COL_ID]] = "id"
rename_map[pdf_full.columns[COL_FECHA]] = "fecha_atencion"
rename_map[pdf_full.columns[COL_GRUPOS]] = "grupos"
pdf = pdf.rename(columns=rename_map)
pdf.columns.tolist()

### Paso 4 · Definición de la cohorte
Conservamos las filas con **al menos un dato** en el bloque L:AB. Verificamos aquí la composición por grupo (control del hallazgo documentado arriba).

In [ ]:
feature_cols = [RENAME_BY_POS[p] for p in FEATURE_POS]
mask_has_data = pdf[feature_cols].notna().any(axis=1)
pdf_cohorte = pdf[mask_has_data].copy()

print(f"Registros en la cohorte (con datos en L:AB): {len(pdf_cohorte):,}\n")
print("Composición por grupo (control):")
comp = pdf_cohorte["grupos"].value_counts(dropna=False)
for g, n in comp.items():
    print(f"   {str(g):<40} {n:>6}  ({100*n/len(pdf_cohorte):.1f}%)")

### Paso 5 · Diagnóstico de calidad de datos
Antes de modelar cuantificamos el **% de llenado** por variable y las **categorías presentes**. Esto justifica las reglas de imputación del notebook 02 (p. ej. `actividad_duracion` y `actividad_frecuencia` tienen alta ausencia).

In [ ]:
llenado = (pdf_cohorte[feature_cols].notna().mean() * 100).round(1).sort_values()
print("% de llenado por variable (ascendente):\n")
print(llenado.to_string())

In [ ]:
cat_cols = ["actividad_tipo", "actividad_duracion", "actividad_frecuencia",
            "estres_altas_cargas", "estres_tecnica_manejo"]
for c in cat_cols:
    print(f"\n=== {c} ===")
    print(pdf_cohorte[c].value_counts(dropna=False).head(10).to_string())

In [ ]:
bin_num_cols = ["habitos_alim_cantidad", "hab_dejar_endulzar", "hab_dejar_reposteria",
                "hab_incorporar_vegetales", "hab_ajustar_carbohidratos", "hab_aumentar_proteina",
                "hab_otros", "med_hipoglicemiantes", "med_hipolipemiantes", "med_ninguno"]
display(pdf_cohorte[bin_num_cols].apply(pd.to_numeric, errors="coerce").describe().T)

### Paso 6 · Persistencia (capa *raw* / bronce)
Guardamos la cohorte cruda como **tabla Delta**. Es el punto de partida versionado y auditable para el resto del pipeline (linaje en Unity Catalog).

In [ ]:
# Tipamos texto como string para evitar problemas al crear el DataFrame Spark
for c in pdf_cohorte.columns:
    if pdf_cohorte[c].dtype == "object":
        pdf_cohorte[c] = pdf_cohorte[c].astype("string")

sdf = spark.createDataFrame(pdf_cohorte)
(sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(RAW_TABLE))
print(f"✅ Tabla guardada: {RAW_TABLE}  ({sdf.count():,} filas)")

In [ ]:
display(spark.table(RAW_TABLE).limit(20))

---
**Resumen del notebook 01:** cohorte definida y perfilada, guardada en `fenotipo_cardio_ingreso_raw`.
➡️ Continúa en **`02_preprocesamiento`** para limpiar, imputar y dejar las variables listas para el modelo.